# AllSides 45 Manual Error Analysis Summary

This notebook summarizes the manually coded qualitative error analysis for the 45 manually annotated AllSides articles.

It:
1. loads the manual error-analysis CSV;
2. checks and cleans `manual_category` labels;
3. counts the manual categories;
4. creates a small number of Plotly visualizations;
5. saves a category-count table


In [1]:
from pathlib import Path

import pandas as pd


pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 100)

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 100)


## 1. Load


In [ ]:
INPUT_PATH = Path("results/allsides_45_manual_error_analysis.csv")
OUTPUT_DIR = Path("results/error_analysis_summary")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH)

print("Rows:", len(df))
print("Columns:", list(df.columns))

df.head()


## 2. Clean and check manual category names

This cell standardizes small spelling variants. It keeps `other` as `other`.


In [ ]:
CATEGORY_RENAMES = {
    "missed:gold_span": "missed_gold_span",
    "no_error_meaningful": "no_substantive_error",
    "no_meaningful_error": "no_substantive_error",
    "label_confusion": "testimony_anecdote_ambiguity",
}

EXPECTED_CATEGORIES = [
    "boundary_issue_only",
    "span_segmentation_issue",
    "noise_or_formatting",
    "no_substantive_error",
    "testimony_anecdote_ambiguity",
    "anecdote_assumption_ambiguity",
    "testimony_assumption_ambiguity",
    "testimony_statistics_ambiguity",
    "statistics_testimony_ambiguity",
    "anecdote_statistics_ambiguity",
    "statistics_anecdote_ambiguity",
    "missed_gold_span",
    "other",
]

df["manual_category"] = (
    df["manual_category"]
    .astype(str)
    .str.strip()
    .replace(CATEGORY_RENAMES)
)

unique_categories = sorted(df["manual_category"].dropna().unique())
unexpected_categories = sorted(set(unique_categories) - set(EXPECTED_CATEGORIES))

print("Unique manual categories:")
for category in unique_categories:
    print("-", category)

if unexpected_categories:
    print("\nUnexpected categories to inspect:")
    for category in unexpected_categories:
        print("-", category)
else:
    print("\nNo unexpected manual_category values found.")


## 3. Count manual error categories

This table is the main quantitative summary of the qualitative error analysis.


In [ ]:
category_counts = (
    df["manual_category"]
    .value_counts()
    .rename_axis("manual_category")
    .reset_index(name="count")
)

category_counts["percentage"] = (
    category_counts["count"] / category_counts["count"].sum() * 100
).round(1)

category_counts


In [ ]:
CATEGORY_LABELS = {
    "boundary_issue_only": "Boundary issue only",
    "span_segmentation_issue": "Span segmentation issue",
    "noise_or_formatting": "Noise/formatting",
    "no_substantive_error": "No substantive error",
    "testimony_anecdote_ambiguity": "Testimony–anecdote ambiguity",
    "anecdote_assumption_ambiguity": "Anecdote–assumption ambiguity",
    "testimony_assumption_ambiguity": "Testimony–assumption ambiguity",
    "testimony_statistics_ambiguity": "Testimony–statistics ambiguity",
    "statistics_testimony_ambiguity": "Statistics–testimony ambiguity",
    "anecdote_statistics_ambiguity": "Anecdote–statistics ambiguity",
    "statistics_anecdote_ambiguity": "Statistics–anecdote ambiguity",
    "missed_gold_span": "Missed gold span",
    "other": "Other",
}

plot_df = category_counts.copy()
plot_df["manual_category_label"] = plot_df["manual_category"].map(CATEGORY_LABELS).fillna(plot_df["manual_category"])
plot_df = plot_df.sort_values("count", ascending=True)

fig = px.bar(
    plot_df,
    x="count",
    y="manual_category_label",
    orientation="h",
    text="count",
    title="Manual error-analysis categories",
    labels={
        "count": "Number of cases",
        "manual_category_label": "Manual category",
    },
    hover_data={
        "manual_category": True,
        "percentage": ":.1f",
        "count": True,
        "manual_category_label": False,
    },
)

fig.update_traces(textposition="outside")
fig.update_layout(
    height=550,
    margin=dict(l=20, r=30, t=60, b=20),
)

fig.show()


## 4. Gold label vs. predicted label

This table is useful for seeing which labels are involved in the inspected disagreement cases. Empty cells indicate that either the gold or predicted side was missing for that sampled row.


In [ ]:
label_df = df.copy()

label_df["gold_label_display"] = label_df["gold_label"].fillna("").replace("", "(none)")
label_df["pred_label_display"] = label_df["pred_label"].fillna("").replace("", "(none)")

gold_pred_table = pd.crosstab(
    label_df["gold_label_display"],
    label_df["pred_label_display"],
)

gold_pred_table


In [ ]:
gold_pred_long = (
    gold_pred_table
    .reset_index()
    .melt(
        id_vars="gold_label_display",
        var_name="pred_label_display",
        value_name="count",
    )
)

fig = px.density_heatmap(
    gold_pred_long,
    x="pred_label_display",
    y="gold_label_display",
    z="count",
    text_auto=True,
    title="Gold label by predicted label in the manual sample",
    labels={
        "pred_label_display": "Predicted label",
        "gold_label_display": "Gold label",
        "count": "Number of cases",
    },
)

fig.update_layout(
    height=500,
    margin=dict(l=20, r=30, t=60, b=20),
)

fig.show()


## 5. Save summary tables

The saved CSV files can be used for thesis tables or appendices.


In [ ]:
category_counts.to_csv(
    OUTPUT_DIR / "manual_error_category_counts.csv",
    index=False,
    encoding="utf-8",
)

gold_pred_table.to_csv(
    OUTPUT_DIR / "manual_error_gold_by_predicted_label.csv",
    encoding="utf-8",
)

cleaned_output_path = OUTPUT_DIR / "allsides_45_error_analysis_manual_sample_cleaned.csv"
df.to_csv(cleaned_output_path, index=False, encoding="utf-8")

print("Saved:")
print("-", OUTPUT_DIR / "manual_error_category_counts.csv")
print("-", OUTPUT_DIR / "manual_error_gold_by_predicted_label.csv")
print("-", cleaned_output_path)


## 6. Short discussion notes

Use these notes as a basis for the thesis subsection. They should be adapted after checking the final counts above.

**Preliminary interpretation.** The manual error analysis suggests that many model–gold disagreements are not complete evidence-type failures. A large portion involves boundary differences or span segmentation, where the model identifies the relevant type of evidence but predicts a shorter or slightly different span than the gold annotation. This supports the use of relaxed span-level evaluation alongside strict BIO evaluation.

The category ambiguities also show that the evidence labels are difficult to distinguish in news-style articles. Anecdote is especially involved in misclassifications because concrete events, factual background, attributed statements, and quantitative information often occur together in the same local context. The analysis therefore suggests that the out-of-domain performance drop reflects not only model error, but also genre-related ambiguity in how evidence is expressed in news reporting.

Finally, noise and formatting cases indicate that some errors are caused by article boilerplate, punctuation, or short fragments. This is relevant for the later large-scale prediction analysis because the unannotated data should be cleaned before inference, and model-derived span counts should be interpreted as approximate predicted evidence mentions rather than exact human-annotated evidence units.
